In [2]:
!pip install transformers datasets sentencepiece torch pandas -q

In [3]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [4]:
model_name = "Helsinki-NLP/opus-mt-en-mk"

In [5]:
print("Вчитување на моделот и токенизаторот...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Вчитување на моделот и токенизаторот...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/799k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(61946, 512, padding_idx=61945)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(61946, 512, padding_idx=61945)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [7]:
print("Вчитување на податоците (Dolly-15k)...")
dataset = load_dataset("databricks/databricks-dolly-15k")
# Земаме само првите 5 примери за тест
small_dataset = dataset["train"].select(range(5))

Вчитување на податоците (Dolly-15k)...


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [8]:
def translate_text(text):
    if not text or text.strip() == "":
        return ""

    # Подготовка на текстот за моделот
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    # Генерирање на преводот
    with torch.no_grad():
        translated_tokens = model.generate(**inputs, max_length=512)

    # Декодирање во чист текст
    result = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return result

In [9]:
translated_examples = []

In [10]:
print("Започнува преводот (ова може да потрае неколку секунди)...")
for i, ex in enumerate(small_dataset):
    print(f"Преведување на пример {i+1}...")

    orig_inst = ex["instruction"]
    orig_resp = ex["response"]
    orig_cont = ex.get("context", "")

    mk_inst = translate_text(orig_inst)
    mk_resp = translate_text(orig_resp)
    mk_cont = translate_text(orig_cont) if orig_cont else ""

    # Ги зачувуваме и оригиналните и преведените вредности за табелата
    translated_examples.append({
        "ID": i + 1,
        "orig_instruction": orig_inst,
        "mk_instruction": mk_inst,
        "orig_response": orig_resp,
        "mk_response": mk_resp,
        "orig_context": orig_cont,
        "mk_context": mk_cont
    })

Започнува преводот (ова може да потрае неколку секунди)...
Преведување на пример 1...
Преведување на пример 2...
Преведување на пример 3...
Преведување на пример 4...
Преведување на пример 5...


In [11]:
df = pd.DataFrame(translated_examples)

In [12]:
pd.set_option("display.max_colwidth", None)

In [13]:
df = df[["ID", "orig_instruction", "mk_instruction", "orig_response", "mk_response"]]

print("\n" + "="*30 + " ГОТОВО! " + "="*30 + "\n")


============================== ГОТОВО! ==============================



In [14]:
print(df.head())

   ID  \
0   1   
1   2   
2   3   
3   4   
4   5   

                                                                               orig_instruction  \
0                                                    When did Virgin Australia start operating?   
1                                                      Which is a species of fish? Tope or Rope   
2                                                Why can camels survive for long without water?   
3  Alice's parents have three daughters: Amy, Jessy, and what’s the name of the third daughter?   
4                                                               When was Tomoaki Komorida born?   

                                                                  mk_instruction  \
0                                    Кога започна операцијата Вирџин Австралија?   
1                                                             Што е вид на риба?   
2                              Зошто камилите можат да преживеат долго без вода?   
3  Родителите 

In [15]:
#df.to_csv("prevedeni_podatoci.csv", index=False)